In [1]:
# %% [markdown]
# # 📊 Парсер котировок MOEX в PostgreSQL (ТОЛЬКО НОВЫЕ ДАННЫЕ)

# %%
# Установка необходимых библиотек
import sys
!{sys.executable} -m pip install pandas moexalgo psycopg2-binary tqdm --upgrade

# %%
# Импорт библиотек
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import logging
from tqdm.notebook import tqdm
import psycopg2
import warnings
warnings.filterwarnings('ignore')

# %%
# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('moex_parser.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# %%
# Конфигурация
class Config:
    # Параметры подключения к БД
    DB_HOST = "109.196.102.91"
    DB_NAME = "default_db"
    DB_USER = "maksv"
    DB_PASSWORD = r"p27Oqwp4Y,X^O^"
    DB_SCHEMA = "public"
    DB_TABLE = "share_prices"
    
    # Путь к файлу с тикерами
    TICKERS_PATH = Path(r"C:\Users\maksv\OneDrive\Рабочий стол\Материалы\Тикеры акций.xlsx")
    
    # Параметры парсинга
    START_DATE = "2000-01-01"
    END_DATE = datetime.now().strftime("%Y-%m-%d")

# %%
# Функция для создания уникального индекса (если его нет)
def ensure_unique_constraint():
    """
    Проверяет наличие уникального ограничения и создает его при необходимости
    """
    conn = None
    try:
        conn = psycopg2.connect(
            host=Config.DB_HOST,
            database=Config.DB_NAME,
            user=Config.DB_USER,
            password=Config.DB_PASSWORD
        )
        cur = conn.cursor()
        
        # Проверяем наличие уникального индекса
        cur.execute(f"""
            SELECT EXISTS (
                SELECT 1
                FROM pg_indexes
                WHERE schemaname = '{Config.DB_SCHEMA}'
                AND tablename = '{Config.DB_TABLE}'
                AND indexdef LIKE '%UNIQUE%ticker%date%'
            )
        """)
        
        has_unique_index = cur.fetchone()[0]
        
        if not has_unique_index:
            logger.info("Создаем уникальный индекс на (ticker, date)...")
            cur.execute(f"""
                CREATE UNIQUE INDEX IF NOT EXISTS idx_ticker_date_unique 
                ON {Config.DB_SCHEMA}.{Config.DB_TABLE} (ticker, date)
            """)
            conn.commit()
            logger.info("✅ Уникальный индекс успешно создан")
        else:
            logger.info("✅ Уникальный индекс уже существует")
        
        return True
        
    except Exception as e:
        logger.error(f"Ошибка при создании индекса: {e}")
        if conn:
            conn.rollback()
        return False
    finally:
        if conn:
            conn.close()

# %%
# Функции для работы с тикерами и ISIN из Excel
def get_tickers_with_isin_from_excel():
    """
    Получение словаря {тикер: isin} из Excel файла
    """
    if not Config.TICKERS_PATH.exists():
        logger.error(f"Файл с тикерами не найден: {Config.TICKERS_PATH}")
        return {}
    
    try:
        df = pd.read_excel(Config.TICKERS_PATH)
        
        # Определяем колонку с тикерами
        ticker_col = None
        possible_ticker_names = ['Ticker', 'ticker', 'Тикер', 'тикер', 'SECID', 'Symbol', 'Код']
        for col in possible_ticker_names:
            if col in df.columns:
                ticker_col = col
                break
        
        # Определяем колонку с ISIN
        isin_col = None
        possible_isin_names = ['ISIN', 'Isin', 'isin', 'Код ISIN', 'ISIN код']
        for col in possible_isin_names:
            if col in df.columns:
                isin_col = col
                break
        
        if ticker_col is None:
            ticker_col = df.columns[0]
        
        # Создаем словарь тикер -> isin
        ticker_isin_map = {}
        
        for idx, row in df.iterrows():
            ticker = str(row[ticker_col]).strip().upper()
            
            if not ticker or ticker == 'NAN' or len(ticker) == 0:
                continue
            
            if isin_col and pd.notna(row[isin_col]):
                isin = str(row[isin_col]).strip().upper()
                isin = ''.join(c for c in isin if c.isalnum())
                ticker_isin_map[ticker] = isin
            else:
                ticker_isin_map[ticker] = None
        
        return ticker_isin_map
    
    except Exception as e:
        logger.error(f"Ошибка при чтении файла с тикерами: {e}")
        return {}

# %%
# Функции для работы с базой данных
def get_db_connection():
    """Создание подключения к БД"""
    try:
        conn = psycopg2.connect(
            host=Config.DB_HOST,
            database=Config.DB_NAME,
            user=Config.DB_USER,
            password=Config.DB_PASSWORD
        )
        return conn
    except Exception as e:
        logger.error(f"Ошибка подключения к БД: {e}")
        raise

def get_last_date_for_ticker(ticker):
    """
    Получение последней даты для тикера
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        cur.execute(f"""
            SELECT MAX(date) 
            FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
            WHERE ticker = %s
        """, (ticker,))
        
        last_date = cur.fetchone()[0]
        return last_date
        
    except Exception as e:
        logger.error(f"Ошибка при получении последней даты для {ticker}: {e}")
        return None
    finally:
        if conn:
            conn.close()

def get_existing_dates_for_ticker(ticker):
    """
    Получение списка существующих дат для тикера
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        cur.execute(f"""
            SELECT date 
            FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
            WHERE ticker = %s
            ORDER BY date
        """, (ticker,))
        
        dates = [row[0] for row in cur.fetchall()]
        return set(dates)
        
    except Exception as e:
        logger.error(f"Ошибка при получении существующих дат для {ticker}: {e}")
        return set()
    finally:
        if conn:
            conn.close()

def save_to_postgresql(df, ticker, isin=None):
    """
    Сохранение только новых данных в PostgreSQL
    """
    if df.empty:
        return 0
    
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        # Получаем существующие даты для этого тикера
        existing_dates = get_existing_dates_for_ticker(ticker)
        
        records_new = []
        
        # Собираем только новые записи
        for _, row in df.iterrows():
            date_value = row['begin']
            if isinstance(date_value, str):
                date_obj = pd.to_datetime(date_value).date()
            elif hasattr(date_value, 'date'):
                date_obj = date_value.date()
            else:
                date_obj = date_value
            
            if date_obj in existing_dates:
                continue
            
            records_new.append((
                ticker,
                isin,
                date_obj,
                float(round(row['open'], 2)),
                float(round(row['high'], 2)),
                float(round(row['low'], 2)),
                float(round(row['close'], 2)),
                float(round(row['avg'], 2)),
                int(row['value'])
            ))
        
        # Вставляем новые записи
        if records_new:
            insert_sql = f"""
                INSERT INTO {Config.DB_SCHEMA}.{Config.DB_TABLE} 
                (ticker, isin, date, open, high, low, close, avg, value)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            
            for record in records_new:
                cur.execute(insert_sql, record)
            
            conn.commit()
            
            # Выводим информацию о добавленных данных
            new_dates = [r[2] for r in records_new]
            print(f"✅ По тикеру {ticker} добавлена новая информация: {len(records_new)} записей с {min(new_dates)} по {max(new_dates)}")
        
        return len(records_new)
        
    except Exception as e:
        logger.error(f"❌ Ошибка при сохранении в БД для тикера {ticker}: {e}")
        if conn:
            conn.rollback()
        return 0
    finally:
        if conn:
            conn.close()

# %%
# Функции для работы с данными MOEX
def get_first_ticker_date(ticker):
    """
    Получение самой первой даты котировок для тикера
    """
    from moexalgo import Ticker
    
    try:
        t = Ticker(ticker)
        
        df_test = t.candles(start="2000-01-01", end="2000-01-10", period='1d')
        
        if not df_test.empty:
            return "2000-01-01"
        
        test_years = [2010, 2015, 2018, 2020, 2021, 2022, 2023, 2024]
        
        for year in test_years:
            df_test = t.candles(start=f"{year}-01-01", end=f"{year}-01-10", period='1d')
            if not df_test.empty:
                return f"{year}-01-01"
        
        return (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")
        
    except Exception as e:
        logger.warning(f"Не удалось определить первую дату для {ticker}: {e}")
        return "2020-01-01"

def fetch_moex_data(ticker, start_date, end_date):
    """
    Получение данных с MOEX за указанный период
    """
    from moexalgo import Ticker
    
    try:
        t = Ticker(ticker)
        df = t.candles(start=start_date, end=end_date, period='1d')
        
        if df.empty:
            return pd.DataFrame()
        
        # Рассчитываем среднюю цену
        df['avg'] = df.apply(
            lambda row: round(row['value'] / row['volume'], 2) if row['volume'] != 0 else row['close'],
            axis=1
        )
        
        # Округляем цены до 2 знаков
        for col in ['open', 'high', 'low', 'close', 'avg']:
            df[col] = df[col].round(2)
        
        # Преобразуем value в int
        df['value'] = df['value'].round(0).astype(int)
        
        # Преобразуем дату
        df['begin'] = pd.to_datetime(df['begin'])
        
        # Сортируем по дате
        df = df.sort_values('begin')
        
        return df
    
    except Exception as e:
        logger.error(f"Ошибка при получении данных для {ticker}: {e}")
        return pd.DataFrame()

# %%
# Основная функция для добавления новых данных
def add_new_ticker_data(ticker, isin=None):
    """
    Добавление новых данных для конкретного тикера
    """
    try:
        # Получаем последнюю дату в БД
        last_date = get_last_date_for_ticker(ticker)
        
        # Определяем период для загрузки
        if last_date is None:
            start_date = get_first_ticker_date(ticker)
        else:
            start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
        
        end_date = Config.END_DATE
        
        if start_date >= end_date:
            return 0
        
        df = fetch_moex_data(ticker, start_date, end_date)
        
        if df.empty:
            return 0
        
        records_added = save_to_postgresql(df, ticker, isin)
        
        return records_added
        
    except Exception as e:
        logger.error(f"Ошибка при добавлении данных для {ticker}: {e}")
        return 0

def add_all_tickers_data(limit=None):
    """
    Добавление новых данных для всех тикеров
    """
    # Сначала создаем уникальный индекс, если его нет
    ensure_unique_constraint()
    
    # Получаем словарь тикер -> isin
    ticker_isin_map = get_tickers_with_isin_from_excel()
    
    if not ticker_isin_map:
        logger.error("Нет тикеров для парсинга")
        return
    
    tickers = list(ticker_isin_map.keys())
    
    if limit:
        tickers = tickers[:limit]
    
    print(f"🔄 Начинаем обновление данных для {len(tickers)} тикеров...")
    
    total_added = 0
    
    for ticker in tqdm(tickers, desc="Обработка тикеров"):
        isin = ticker_isin_map.get(ticker)
        
        try:
            records_added = add_new_ticker_data(ticker, isin)
            total_added += records_added
            
            if records_added == 0:
                print(f"ℹ️ По тикеру {ticker} нет новых данных")
            
        except Exception as e:
            logger.error(f"✗ Ошибка при обработке {ticker}: {e}")
            print(f"❌ Ошибка при обработке {ticker}")
    
    print(f"\n✅ Обновление завершено. Всего добавлено {total_added} новых записей")
    
    return total_added

# %%
# ЗАПУСК ДОБАВЛЕНИЯ ДАННЫХ
if __name__ == "__main__":
    
    print("📊 ДОБАВЛЕНИЕ НОВЫХ ДАННЫХ MOEX")
    print("="*60)
    
    # ЗАПУСК - добавляем только новые данные
    total_added = add_all_tickers_data(limit=None)  # Можно указать limit=5 для теста


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2026-03-01 13:15:09,160 - __main__ - INFO - ✅ Уникальный индекс уже существует


📊 ДОБАВЛЕНИЕ НОВЫХ ДАННЫХ MOEX
🔄 Начинаем обновление данных для 262 тикеров...


Обработка тикеров:   0%|          | 0/262 [00:00<?, ?it/s]

2026-03-01 13:15:10,818 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/securities/ABIO.json "HTTP/1.1 200 OK"
2026-03-01 13:15:11,295 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ABIO/candles.json?from=2026-02-27&till=2026-03-01&interval=24&start=0 "HTTP/1.1 200 OK"
2026-03-01 13:15:11,367 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ABIO/candles.json?from=2026-02-27&till=2026-03-01&interval=24&start=3 "HTTP/1.1 200 OK"


...

✅ По тикеру ZAYM добавлена новая информация: 3 записей с 2026-02-27 по 2026-03-01


2026-03-01 13:21:19,217 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/securities/ZILL.json "HTTP/1.1 200 OK"
2026-03-01 13:21:19,714 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ZILL/candles.json?from=2026-02-27&till=2026-03-01&interval=24&start=0 "HTTP/1.1 200 OK"
2026-03-01 13:21:19,771 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ZILL/candles.json?from=2026-02-27&till=2026-03-01&interval=24&start=2 "HTTP/1.1 200 OK"


✅ По тикеру ZILL добавлена новая информация: 2 записей с 2026-02-27 по 2026-02-28


2026-03-01 13:21:20,621 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/securities/ZVEZ.json "HTTP/1.1 200 OK"
2026-03-01 13:21:21,097 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ZVEZ/candles.json?from=2026-02-27&till=2026-03-01&interval=24&start=0 "HTTP/1.1 200 OK"
2026-03-01 13:21:21,192 - httpx - INFO - HTTP Request: GET https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ZVEZ/candles.json?from=2026-02-27&till=2026-03-01&interval=24&start=2 "HTTP/1.1 200 OK"


✅ По тикеру ZVEZ добавлена новая информация: 2 записей с 2026-02-27 по 2026-02-28

✅ Обновление завершено. Всего добавлено 736 новых записей
